In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("라이브러리 로드 완료!")

라이브러리 로드 완료!


In [2]:
df = pd.read_csv('../data/raw/stocks/kospi200_5y.csv')
df.columns = df.columns.str.lower()

print(f"데이터 shape: {df.shape}")
print(f"\n컬럼 목록: {df.columns.tolist()}")
print(f"\n처음 5행:")
df.head()

데이터 shape: (230938, 8)

컬럼 목록: ['date', 'open', 'high', 'low', 'close', 'volume', 'change', 'ticker']

처음 5행:


,date,open,high,low,close,volume,change,ticker
0,2021-03-31,82400,82700,81400,81400,17240518,-0.009732,5930
1,2021-04-01,82500,83000,82000,82900,18676461,0.018428,5930
2,2021-04-02,84000,85200,83900,84800,22997538,0.022919,5930
3,2021-04-05,85800,86000,84800,85400,16255990,0.007075,5930
4,2021-04-06,86200,86200,85100,86000,19042023,0.007026,5930


In [3]:
print("=== 기본 통계 ===")
print(f"총 행 수:    {len(df):,}")
print(f"총 종목 수:  {df['ticker'].nunique():,}")
print(f"수집 기간:   {df['date'].min()} ~ {df['date'].max()}")
print(f"\n결측치 현황:")
print(df.isnull().sum())

=== 기본 통계 ===
총 행 수:    230,938
총 종목 수:  202
수집 기간:   2021-03-31 ~ 2026-03-30

결측치 현황:
date       0
open       0
high       0
low        0
close      0
volume     0
change    24
ticker     0
dtype: int64


In [4]:
samsung = df[df['ticker'] == '005930'].copy()
samsung['date'] = pd.to_datetime(samsung['date'])
samsung = samsung.sort_values('date')

print(f"삼성전자 데이터: {len(samsung)}행")
print(samsung.head(3))

삼성전자 데이터: 0행
Empty DataFrame
Columns: [date, open, high, low, close, volume, change, ticker]
Index: []


In [5]:
print("ticker 샘플 값:")
print(df['ticker'].head(10).tolist())
print(f"\nticker 데이터 타입: {df['ticker'].dtype}")

ticker 샘플 값:
[5930, 5930, 5930, 5930, 5930, 5930, 5930, 5930, 5930, 5930]

ticker 데이터 타입: object


In [6]:
df['ticker'] = df['ticker'].astype(str).str.zfill(6)

print("변환 후 ticker 샘플:")
print(df['ticker'].head(10).tolist())

변환 후 ticker 샘플:
['005930', '005930', '005930', '005930', '005930', '005930', '005930', '005930', '005930', '005930']


In [7]:
samsung = df[df['ticker'] == '005930'].copy()
samsung['date'] = pd.to_datetime(samsung['date'])
samsung = samsung.sort_values('date')

print(f"삼성전자 데이터: {len(samsung)}행")

fig = px.line(
    samsung,
    x='date',
    y='close',
    title='삼성전자 종가 추이 (5년)',
    labels={'close': '종가 (원)', 'date': '날짜'}
)
fig.update_layout(height=400)
fig.show()

삼성전자 데이터: 1224행


In [8]:
fig = make_subplots(rows=2, cols=1,
                    subplot_titles=('종가', '거래량'),
                    row_heights=[0.7, 0.3])

fig.add_trace(
    go.Scatter(x=samsung['date'], y=samsung['close'],
               name='종가', line=dict(color='#378ADD')),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=samsung['date'], y=samsung['volume'],
           name='거래량', marker_color='#1D9E75'),
    row=2, col=1
)

fig.update_layout(height=600, title='삼성전자 종가 + 거래량')
fig.show()

In [9]:
print(f"거래량 최솟값: {samsung['volume'].min()}")
print(f"거래량 최댓값: {samsung['volume'].max()}")
print(f"거래량 결측치: {samsung['volume'].isnull().sum()}")
print(samsung['volume'].head())

거래량 최솟값: 5767902
거래량 최댓값: 89427954
거래량 결측치: 0
0    17240518
1    18676461
2    22997538
3    16255990
4    19042023
Name: volume, dtype: int64


In [10]:
df['date'] = pd.to_datetime(df['date'])
df_sorted = df.sort_values(['ticker', 'date'])

returns = []
for ticker, group in df_sorted.groupby('ticker'):
    if len(group) > 0:
        first_close = group['close'].iloc[0]
        last_close  = group['close'].iloc[-1]
        ret = (last_close - first_close) / first_close * 100
        returns.append({'ticker': ticker, 'return_pct': round(ret, 2)})

returns_df = pd.DataFrame(returns).sort_values('return_pct', ascending=False)

print("=== 5년 수익률 상위 10개 종목 ===")
print(returns_df.head(10).to_string(index=False))
print("\n=== 5년 수익률 하위 10개 종목 ===")
print(returns_df.tail(10).to_string(index=False))

=== 5년 수익률 상위 10개 종목 ===
ticker  return_pct
267260     4062.29
007660     3443.57
298040     3092.60
012450     3039.55
000150     2126.23
042700     2035.25
103590     1866.85
079550     1608.59
071970     1530.31
003230     1219.10

=== 5년 수익률 하위 10개 종목 ===
ticker  return_pct
051910      -60.00
251270      -61.43
302440      -64.90
323410      -65.04
011170      -71.36
377300      -73.68
036570      -74.57
018880      -76.47
051900      -84.46
361610      -85.70


In [11]:
fig = px.histogram(
    returns_df,
    x='return_pct',
    nbins=50,
    title='코스피 200 종목 5년 수익률 분포',
    labels={'return_pct': '수익률 (%)'},
    color_discrete_sequence=['#7F77DD']
)
fig.add_vline(x=0, line_dash='dash', line_color='red',
              annotation_text='기준선 0%')
fig.update_layout(height=400)
fig.show()

In [12]:
samsung = samsung.sort_values('date').copy()

# 이동평균선
samsung['ma5']  = samsung['close'].rolling(window=5).mean()
samsung['ma20'] = samsung['close'].rolling(window=20).mean()
samsung['ma60'] = samsung['close'].rolling(window=60).mean()

# 일별 수익률
samsung['daily_return'] = samsung['close'].pct_change() * 100

fig = px.line(samsung, x='date',
              y=['close', 'ma5', 'ma20', 'ma60'],
              title='삼성전자 이동평균선',
              labels={'value': '가격 (원)', 'date': '날짜'})
fig.update_layout(height=500)
fig.show()

In [13]:
# 전체 데이터에 ticker 앞자리 0 복원 + 날짜 변환 적용
df['ticker'] = df['ticker'].astype(str).str.zfill(6)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['ticker', 'date'])

# 종목별 이동평균 + 일별수익률 추가
df['ma5']  = df.groupby('ticker')['close'].transform(lambda x: x.rolling(5).mean())
df['ma20'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(20).mean())
df['ma60'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(60).mean())
df['daily_return'] = df.groupby('ticker')['close'].transform(lambda x: x.pct_change() * 100)

# 저장
df.to_csv('../data/processed/kospi200_processed.csv', index=False, encoding='utf-8-sig')

print(f"저장 완료!")
print(f"shape: {df.shape}")
print(f"\n컬럼 목록: {df.columns.tolist()}")
print(f"\n결측치 현황:")
print(df.isnull().sum())

저장 완료!
shape: (230938, 12)

컬럼 목록: ['date', 'open', 'high', 'low', 'close', 'volume', 'change', 'ticker', 'ma5', 'ma20', 'ma60', 'daily_return']

결측치 현황:
date                0
open                0
high                0
low                 0
close               0
volume              0
change             24
ticker              0
ma5               800
ma20             3799
ma60            11759
daily_return      200
dtype: int64


In [ ]:
## shape: (230,938 행 × 12 컬럼)  ✅
## 결측치: change 24개            ✅ 무시 가능
## ma5/ma20/ma60 결측치           ✅ 정상 (초반 윈도우 기간)
## daily_return 200개             ✅ 정상 (각 종목 첫날)

# ✅ 230,938행 주가 데이터 확인
# ✅ 삼성전자 종가 추이 시각화
# ✅ 코스피 200 수익률 분포 분석
# ✅ 이동평균선 (MA5, MA20, MA60) 추가
# ✅ 전처리 데이터 저장
#    → data/processed/kospi200_processed.csv

## shape: (230,938 행 × 12 컬럼)  ✅
## 결측치: change 24개            ✅ 무시 가능
## ma5/ma20/ma60 결측치           ✅ 정상 (초반 윈도우 기간)
## daily_return 200개             ✅ 정상 (각 종목 첫날)

# ✅ 230,938행 주가 데이터 확인
# ✅ 삼성전자 종가 추이 시각화
# ✅ 코스피 200 수익률 분포 분석
# ✅ 이동평균선 (MA5, MA20, MA60) 추가
# ✅ 전처리 데이터 저장
#    → data/processed/kospi200_processed.csv